In [3]:
!pip install opencv-python
!pip install numpy
!pip install pytesseract
!pip install csv
!pip install pandas
!pip install re
!pip install os

ERROR: Could not find a version that satisfies the requirement csv (from versions: none)
ERROR: No matching distribution found for csv


ERROR: Could not find a version that satisfies the requirement re (from versions: none)
ERROR: No matching distribution found for re
ERROR: Could not find a version that satisfies the requirement os (from versions: none)
ERROR: No matching distribution found for os


In [ ]:
import cv2
import numpy as np
import pytesseract
import datetime
import csv
import pandas as pd
import re
import os

pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'

def normalize_ocr_time(raw):
    """
    Clean OCR noise and extract HH:MM:SS.
    Output -> MM:SS from the middle + last component.
    """
    if not raw:
        return None

    # Remove garbage except digits and colon
    raw = raw.strip()
    raw = re.sub(r"[^0-9:]", "", raw)

    # Enforce HH:MM:SS by extracting last 3 numeric groups
    parts = re.findall(r"\d+", raw)

    if len(parts) < 3:
        return None

    # Consider last 3 groups as HH, MM, SS
    h, mm, ss = parts[-3], parts[-2], parts[-1]

    try:
        mm_i = int(mm)
        ss_i = int(ss)
        if mm_i > 59 or ss_i > 59:
            return None  # definitely a mis-OCR
        return f"{mm_i:02d}:{ss_i:02d}"
    except:
        return None


def normalize_csv_time(raw):
    """
    Accept:
      - MM:SS
      - MM:SS.d
      - M:SS
      - SS decimal handling
    Convert -> MM:SS with rounding + carry if needed.
    """
    if not raw:
        return None

    # Remove surrounding spaces + strip date if present
    raw = raw.strip()
    raw = re.sub(r'\d{4}-\d{2}-\d{2}', '', raw)

    # Extract digits + colon + decimals
    raw = re.sub(r"[^0-9:\.]", "", raw)

    parts = raw.split(":")
    if len(parts) != 2:
        return None

    try:
        mm = int(parts[0])
        sec_float = float(parts[1])

        ss = round(sec_float)  # handle .d or .dd

        # carry if >= 60
        mm += ss // 60
        ss = ss % 60

        return f"{mm:02d}:{ss:02d}"
    except:
        return None


# -------------------------------------------------------------------
# VIDEO PROCESSING: extract clocks & split eye videos
# -------------------------------------------------------------------
def func(vid_path):
    cap = cv2.VideoCapture(vid_path)
    clocks = []
    dict_seconds = {}

    ret, frame = cap.read()
    if not ret:
        return [], {}

    # Eye crop dimensions
    EYE_TOP, EYE_BOTTOM = 340, 640
    EYE_LEFT, EYE_RIGHT = 80, 880
    eye_h = EYE_BOTTOM - EYE_TOP
    eye_w = (EYE_RIGHT - EYE_LEFT) // 2

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    right_writer = cv2.VideoWriter(f"{vid_path[:-4]}_right.mp4", fourcc, 30, (eye_w, eye_h))
    left_writer  = cv2.VideoWriter(f"{vid_path[:-4]}_left.mp4",  fourcc, 30, (eye_w, eye_h))

    frame_idx = 0
    prev_clock_gray = None
    prev_time_norm = None

    cap.set(cv2.CAP_PROP_POS_FRAMES, 0)

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # -------------------------------
        # EYE CROPPING
        # -------------------------------
        eyes_crop = frame[EYE_TOP:EYE_BOTTOM, EYE_LEFT:EYE_RIGHT]
        mid = eyes_crop.shape[1] // 2
        right_writer.write(cv2.cvtColor(eyes_crop[:, :mid], cv2.COLOR_BGR2RGB))
        left_writer.write(cv2.cvtColor(eyes_crop[:, mid:], cv2.COLOR_BGR2RGB))

        # -------------------------------
        # CLOCK OCR (two possible locations)
        # -------------------------------
        # First crop
        clock_crop_1 = frame[1030:1050, 1810:1870]
        # Second crop
        clock_crop_2 = frame[1030:1050, 1757:1817]

        # Convert first crop
        clock_gray = cv2.cvtColor(clock_crop_1, cv2.COLOR_BGR2GRAY)

        # -------------------------------
        # OCR every 3 frames
        # -------------------------------
        if frame_idx % 3 == 0:

            # Try OCR on first crop
            # cv2.imshow('Clock Crop', clock_gray)
            # cv2.waitKey(1)

            def run_ocr(crop):
                g = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
                p = cv2.resize(g, None, fx=8, fy=8)
                p = cv2.GaussianBlur(p, (3,3), 0)
                _, p = cv2.threshold(p, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
                raw = pytesseract.image_to_string(p, config="--psm 7 -c tessedit_char_whitelist=0123456789:")
                return normalize_ocr_time(raw)

            # ------------ Try first coordinates ------------
            norm = run_ocr(clock_crop_1)

            # ------------ If first fails, try second --------
            if not norm:
                norm = run_ocr(clock_crop_2)

            # ------------ If still nothing, skip ------------
            if norm and norm != prev_time_norm:
                clocks.append((frame_idx, norm))
                dict_seconds[len(clocks)-1] = frame_idx
                prev_time_norm = norm

        frame_idx += 1
        
    print("Extracted clocks:", clocks)
    print("Clock dict seconds:", dict_seconds)

    right_writer.release()
    left_writer.release()
    cap.release()
    return clocks, dict_seconds


# -------------------------------------------------------------------
# MATCH SWITCH CLOCKS TO VIDEO FRAMES
# -------------------------------------------------------------------
def func2(vid_path, clocks, dict_seconds, path):
    data_df = pd.read_csv("./equiluminance.csv", header=None)
    match = data_df[data_df[0] == path]
    if match.empty:
        print(f"Warning: no row found for path: {path}. Skipping this file.")
        return []

    row = match.iloc[0]
    row_str = ",".join(row.astype(str))

    # Clocks start from column 4
    clocks_list = row_str.split(",")[4:]

    # Normalize switch clocks from CSV
    switch_clocks = [
        normalize_csv_time(re.sub(r'\d{4}-\d{2}-\d{2}', '', x))
        for x in clocks_list
    ]
    switch_clocks = [x for x in switch_clocks if x]
    print("Switch clocks:", switch_clocks)

    # ---------------------------------------------
    # IMPORTANT FIX: map time string → actual frame number
    # ---------------------------------------------
    detected_times = {c[1]: c[0] for c in clocks}     # FIXED
    print("Detected times:", detected_times)

    # ---------------------------------------------
    # Convert switch clocks to closest true frame
    # ---------------------------------------------
    frame_ests = []
    for sw in switch_clocks:

        if sw in detected_times:
            frame_ests.append(detected_times[sw])
            continue

        # fallback: find closest time
        sw_m, sw_s = map(int, sw.split(":"))
        sw_total = sw_m * 60 + sw_s

        best_frame = None
        best_diff = float("inf")

        for det_time, det_frame in detected_times.items():
            d_m, d_s = map(int, det_time.split(":"))
            d_total = d_m * 60 + d_s
            diff = abs(sw_total - d_total)
            if diff < best_diff:
                best_diff = diff
                best_frame = det_frame

        frame_ests.append(best_frame)

    print("Frame estimates:", frame_ests)

    # ---------------------------------------------
    # Compute actual switch frame offsets
    # ---------------------------------------------
    clock_switch_frames = [frame_ests[0]]
    for i in range(1, len(frame_ests)):
        clock_switch_frames.append(frame_ests[i])

    start_frame = clock_switch_frames[0]
    color_switch_frames = [f for f in clock_switch_frames]
    # color_switch_frames = [f - start_frame for f in clock_switch_frames]

    return color_switch_frames


# -------------------------------------------------------------------
# CSV Helpers
# -------------------------------------------------------------------
def read_csv(file_path):
    with open(file_path, mode='r', newline='') as f:
        return [row for row in csv.reader(f)]


def write_csv(file_path, data):
    with open(file_path, mode='w', newline='') as f:
        csv.writer(f).writerows(data)

# -------------------------------------------------------------------
# MAIN LOOP
# -------------------------------------------------------------------
folder_path = "./eq"  # <-- change this to your folder

# Get all video file names from the folder
paths = os.listdir(folder_path)
video_extensions = ('.mp4', '.avi', '.mov', '.mkv')
paths = [p for p in paths if p.lower().endswith(video_extensions)]

processed_entries = []

for p in paths:
    name, ext = os.path.splitext(p)

    # Trim last 11 characters from the name only for reporting
    trimmed_name = name[:-11] if len(name) > 11 else name

    # Store both versions
    processed_entries.append((trimmed_name, p))  # (short_name, full_filename)

print("Processed entries:", processed_entries)

for trimmed, full_filename in processed_entries:
    full_path = f"{folder_path}/{full_filename}"
    print("Processing:", trimmed, "| Using file:", full_filename)

    clocks, dict_seconds = func(full_path)
    color_switch_frames = func2(full_path, clocks, dict_seconds, trimmed)

    output_row = [trimmed] + color_switch_frames
    print(f"Color switch frames: {output_row}")

    data = read_csv('./timestamps_eq.csv')
    data.append(output_row)
    write_csv('./timestamps_eq.csv', data)

    print("Done.\n")

Processed entries: [('eq_neq_001', 'eq_neq_001.mp4'), ('eq_neq_002', 'eq_neq_002.mp4'), ('eq_neq_003', 'eq_neq_003.mp4'), ('eq_neq_004', 'eq_neq_004.mp4'), ('eq_neq_005', 'eq_neq_005.mp4'), ('neq_eq_001', 'neq_eq_001.mp4'), ('neq_eq_002', 'neq_eq_002.mp4'), ('neq_eq_003', 'neq_eq_003.mp4'), ('neq_eq_004', 'neq_eq_004.mp4'), ('neq_eq_005', 'neq_eq_005.mp4')]
Processing: eq_neq_001 | Using file: eq_neq_001.mp4
Extracted clocks: [(0, '41:20'), (21, '41:21'), (48, '41:22'), (78, '41:23'), (108, '41:24'), (138, '41:25'), (168, '41:26'), (198, '41:27'), (228, '41:28'), (258, '41:29'), (288, '41:30'), (318, '41:31'), (348, '41:32'), (381, '41:33'), (411, '41:34'), (438, '41:35'), (468, '41:36'), (498, '41:37'), (528, '41:38'), (561, '41:39'), (591, '41:40'), (618, '41:41'), (648, '41:42'), (678, '41:43'), (708, '41:44'), (738, '41:45'), (768, '41:46'), (798, '41:47'), (828, '41:48'), (858, '41:49'), (888, '41:50'), (918, '41:51'), (948, '41:52'), (978, '41:53'), (1008, '41:54'), (1038, '41:55